# Tinjau query tambahan
Notebook ini mengekstrak PAA yang sudah tersimpan dan membaca keputusan seleksi. Tidak melakukan pencarian SerpApi atau panggilan Gemini. Daftar query batch utama pertama tetap dipertahankan.
Seleksi awal dilakukan asisten, bukan penilai manusia independen. Periksa alasan pada `needs_review` dan `excluded`. Setelah respons utama bertambah, menjalankan ulang notebook bisa menambahkan kandidat `pending` yang belum ditinjau.

In [17]:
import sys
from pathlib import Path
from collections import Counter
from html import escape
from IPython.display import HTML, display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/prepare_query_expansion.py').exists())
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
from prepare_query_expansion import prepare, read_json, read_rows
config = read_json(ROOT / 'configs/query_expansion_01.json')
OUTPUT = ROOT / 'data/interim/query_expansion' / config['expansion_id']

def show(rows, columns, limit=100):
    head = ''.join('<th>' + escape(c) + '</th>' for c in columns)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(r.get(c, ''))) + '</td>' for c in columns) + '</tr>' for r in rows[:limit])
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))
    print(f'Ditampilkan {min(limit, len(rows))} dari {len(rows)} baris.')

In [20]:
# Jalankan ulang untuk memuat PAA tersimpan terbaru; keputusan lama tetap dibaca.
candidates = prepare(config)
print('Diterima per domain:', dict(Counter(r['domain'] for r in candidates if r['selection_status'] == 'accepted')))
DOMAIN = ''  # kosong untuk semua; atau kesehatan / keuangan / teknologi
STATUS = 'needs_review'  # accepted / needs_review / excluded / pending / kosong
selected = [r for r in candidates if (not DOMAIN or r['domain'] == DOMAIN) and (not STATUS or r['selection_status'] == STATUS)]
show(selected, ['query_text', 'domain', 'selection_status', 'selection_reason', 'retrieval_query', 'source_batch', 'paa_depth'])

Kemunculan PAA: 324 | query unik dengan domain: 316
Kandidat baru: 309 {'kesehatan': 121, 'keuangan': 109, 'teknologi': 79}
Status: {'accepted': 127, 'excluded': 33, 'needs_review': 23, 'pending': 126}
Output: D:\Kuliah\TA\final-assignment\data\interim\query_expansion\query_expansion_01
Diterima per domain: {'kesehatan': 62, 'keuangan': 34, 'teknologi': 31}


query_text,domain,selection_status,selection_reason,retrieval_query,source_batch,paa_depth
Cara mengecek apakah kita dapat bantuan BPJS Ketenagakerjaan?,keuangan,needs_review,Jenis program bantuan belum disebutkan.,bpjs ketenagakerjaan,paa_expansion_01,1
BPJS Ketenagakerjaan dicairkan berapa?,keuangan,needs_review,Program manfaat dan kondisi kepesertaan tidak disebutkan.,bpjs ketenagakerjaan,paa_expansion_01,1
KUR BRI 2026 pinjaman 50 juta angsuran berapa?,keuangan,needs_review,Tenor dan skema pinjaman belum tersedia.,kur bri 2026,paa_expansion_01,1
Siapa investor nomor 1 di Indonesia?,keuangan,needs_review,Kriteria nomor satu tidak disebutkan.,investor,paa_expansion_01,1
Investasi 1 juta per bulan dapat berapa?,keuangan,needs_review,"Instrumen, jangka waktu, dan asumsi imbal hasil belum disebutkan.",investor,paa_expansion_01,1
Bank yang paling aman di Indonesia bank apa?,keuangan,needs_review,Kriteria keamanan belum jelas.,bank,paa_expansion_01,1
Pinjam uang 50 juta di bank BCA cicilan berapa?,keuangan,needs_review,Produk dan tenor pinjaman belum disebutkan.,bank,paa_expansion_01,1
Pinjam 100 juta di Mandiri angsuran berapa?,keuangan,needs_review,Produk dan tenor pinjaman belum disebutkan.,bank mandiri,paa_expansion_01,1
Driver itu artinya apa?,teknologi,needs_review,Driver dapat berarti perangkat lunak atau pengemudi; teks tidak menentukan konteks.,driver,paa_expansion_01,1
Arti drive apa?,teknologi,needs_review,Drive memiliki beberapa makna; konteks komputasi belum eksplisit.,driver,paa_expansion_01,1


Ditampilkan 23 dari 23 baris.


## Review asisten berdasarkan rubrik

Snapshot ini mencakup **149 query: 23 needs_review dan 126 pending**. Hasil penilaian: **119 accepted dan 30 excluded**. Ini penilaian asisten, bukan hasil penilaian manusia independen atau pengujian Gemini.

Patokan: intent dapat dipahami dari teks, relevan dengan domain, berbahasa Indonesia yang dapat dipahami, meminta informasi yang dapat dibahas artikel, dan tidak menduplikasi kebutuhan informasi query terpilih. Pertanyaan umum, perbandingan terbaik, nominal yang bergantung skenario, serta premis yang mungkin salah tetap diperbolehkan. Penerimaan tidak membuktikan keamanan obat, kebenaran premis, atau keberhasilan grounding.

Deduplikasi manual memakai objek, kebutuhan informasi, dan batas eksplisit. Perubahan produk, nominal, kondisi pengguna, waktu historis, atau spesifikasi dipertahankan jika mengubah kebutuhan informasi; variasi susunan kata dengan kebutuhan sama dipilih satu wakil. Query utama atau query yang sudah accepted diutamakan, lalu wakil dalam snapshot ini. `duplicate_of` mencatat query_id wakil dan alasan menyebut teksnya. Ini keputusan operasional asisten yang dapat dikoreksi peneliti, bukan pengukuran kemiripan otomatis.

Jalankan sel dari atas: sel daftar keputusan berikut hanya mendefinisikan penilaian; sel penerapan sesudahnya memperbarui CSV secara lokal. **Tidak ada panggilan SerpApi/Gemini.** Sel penerapan mempertahankan keputusan final dari peneliti; koreksi manual di bagian berikutnya tetap dapat mengganti hasil asisten. Mengulang penerapan tanpa perubahan tidak memperbarui timestamp keputusan asisten.

Daftar ini adalah snapshot tetap: query baru yang muncul kemudian tidak otomatis diterima. Semua teks PAA asli dipertahankan. Keputusan accepted/excluded yang sudah ada di luar 149 query tidak ditinjau ulang pada langkah ini, sehingga audit konsistensi seluruh pool masih diperlukan sebelum pembekuan dataset.


In [21]:
# Keputusan asisten untuk snapshot 149 query needs_review/pending.
# Setiap entri dapat diedit; keputusan manual peneliti di bawah tetap diutamakan.
assistant_decisions = {'query_b92a983329ccb3bc27fcde28': {'query_text': 'Cara mengecek apakah kita dapat bantuan BPJS '
                                                  'Ketenagakerjaan?',
                                    'domain': 'keuangan',
                                    'previous_status': 'needs_review',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta prosedur mengecek kelayakan bantuan terkait BPJS '
                                              'Ketenagakerjaan; jenis bantuan dapat dijelaskan sebagai '
                                              'pilihan tanpa mengasumsikan penerima pasti berhak.'},
 'query_7407efade809c8320e703951': {'query_text': 'BPJS Ketenagakerjaan dicairkan berapa?',
                                    'domain': 'keuangan',
                                    'previous_status': 'needs_review',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta penjelasan nilai pencairan manfaat BPJS '
                                              'Ketenagakerjaan; program dan kondisi peserta memengaruhi '
                                              'perhitungan, tetapi intent keuangan jelas.'},
 'query_9f5937618dd5bc033072a29f': {'query_text': 'KUR BRI 2026 pinjaman 50 juta angsuran berapa?',
                                    'domain': 'keuangan',
                                    'previous_status': 'needs_review',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta perhitungan angsuran KUR BRI 2026 sebesar 50 juta; '
                                              'tenor yang belum disebutkan dapat disajikan sebagai beberapa '
                                              'skenario.'},
 'query_38b892f51f9a9a1781c667a7': {'query_text': 'Siapa investor nomor 1 di Indonesia?',
                                    'domain': 'keuangan',
                                    'previous_status': 'needs_review',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta perbandingan investor terkemuka di Indonesia; '
                                              'kriteria peringkat perlu dijelaskan dalam jawaban dan tidak '
                                              'dianggap ada satu peringkat resmi yang pasti.'},
 'query_85f0b7e3de297094a7e1fed9': {'query_text': 'Investasi 1 juta per bulan dapat berapa?',
                                    'domain': 'keuangan',
                                    'previous_status': 'needs_review',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta penjelasan hasil investasi rutin satu juta per bulan; '
                                              'instrumen, durasi, dan imbal hasil menjadi asumsi '
                                              'perhitungan, bukan syarat kejelasan intent.'},
 'query_e5f48338d4d39e5767a5c306': {'query_text': 'Bank yang paling aman di Indonesia bank apa?',
                                    'domain': 'keuangan',
                                    'previous_status': 'needs_review',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta perbandingan keamanan bank di Indonesia; jawaban '
                                              'dapat menjelaskan indikator keamanan tanpa menjamin satu bank '
                                              'selalu paling aman.'},
 'query_a5735329526fdbead08de9fb': {'query_text': 'Pinjam uang 50 juta di bank BCA cicilan berapa?',
                                    'domain': 'keuangan',
                                    'previous_status': 'needs_review',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta perhitungan cicilan pinjaman BCA sebesar 50 juta; '
                                              'produk dan tenor dapat dijelaskan sebagai skenario.'},
 'query_5b0eeae39852833981a0c4a4': {'query_text': 'Pinjam 100 juta di Mandiri angsuran berapa?',
                                    'domain': 'keuangan',
                                    'previous_status': 'needs_review',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta perhitungan angsuran pinjaman Mandiri sebesar 100 '
                                              'juta; produk dan tenor memengaruhi angka, tetapi kebutuhan '
                                              'informasinya jelas.'},
 'query_368e6cc9ebf78dc67464f043': {'query_text': 'Driver itu artinya apa?',
                                    'domain': 'teknologi',
                                    'previous_status': 'needs_review',
                                    'language': 'id',
                                    'status': 'excluded',
                                    'reason': 'Objek driver ambigu antara perangkat lunak dan pengemudi; '
                                              'domain teknologi hanya dapat dipastikan dengan konteks '
                                              'tambahan.'},
 'query_fda5f348eb5cf8c0ebb4eff6': {'query_text': 'Arti drive apa?',
                                    'domain': 'teknologi',
                                    'previous_status': 'needs_review',
                                    'language': 'id',
                                    'status': 'excluded',
                                    'reason': 'Kata drive mempunyai makna umum dan komputasi; teks tidak '
                                              'menentukan objek teknologi sehingga memerlukan tebakan '
                                              'konteks.'},
 'query_4c0b52332b58ab2a4cfee4ce': {'query_text': 'Driver apa ya?',
                                    'domain': 'teknologi',
                                    'previous_status': 'needs_review',
                                    'language': 'id',
                                    'status': 'excluded',
                                    'reason': 'Objek driver ambigu antara perangkat lunak dan pengemudi; '
                                              'pertanyaan tidak menyebut konteks perangkat lunak.'},
 'query_830f1970d267fc383928788b': {'query_text': 'Bagaimana cara masuk ke Chrome?',
                                    'domain': 'teknologi',
                                    'previous_status': 'needs_review',
                                    'language': 'id',
                                    'status': 'excluded',
                                    'reason': 'Masuk ke Chrome dapat berarti membuka aplikasi atau login '
                                              'akun; prosedur yang diminta tidak dapat dibedakan dari teks '
                                              'saja.'},
 'query_267f1d66241055641e665459': {'query_text': 'Apa obat mata yang paling bagus?',
                                    'domain': 'kesehatan',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta informasi pilihan obat mata secara umum; kondisi mata '
                                              'dan kriteria pemilihan dapat dijelaskan tanpa menyatakan satu '
                                              'obat cocok untuk semua.'},
 'query_98634bc2cdbfa39ae6ad9747': {'query_text': 'Salep mata apa yang aman untuk ibu hamil?',
                                    'domain': 'kesehatan',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta informasi keamanan salep mata pada kehamilan; objek '
                                              'obat dan kelompok pengguna jelas.'},
 'query_749948674c8ae41705de2a10': {'query_text': 'Bolehkah anak 2 tahun diberi obat tetes mata?',
                                    'domain': 'kesehatan',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta penjelasan kelayakan penggunaan tetes mata untuk anak '
                                              'usia dua tahun; usia dan tindakan yang ditanyakan eksplisit.'},
 'query_b6ccc742559073f43e37722c': {'query_text': 'Apa obat mata gatal merah?',
                                    'domain': 'kesehatan',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta pilihan penanganan mata gatal dan merah; gejala dan '
                                              'kebutuhan informasi kesehatan jelas.'},
 'query_ac06274fb6f4463e7e481ac7': {'query_text': 'Apa obat flu yang bagus?',
                                    'domain': 'kesehatan',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta informasi pilihan obat flu; kata bagus diperlakukan '
                                              'sebagai kebutuhan perbandingan, bukan klaim obat terbaik '
                                              'universal.'},
 'query_813a82f245fc561fdb508758': {'query_text': 'Gimana biar flu cepat hilang?',
                                    'domain': 'kesehatan',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta langkah meredakan flu; bahasa percakapan tetap jelas '
                                              'dan tidak memerlukan penambahan objek.'},
 'query_e959d5cd0dcd4576b9c6f7da': {'query_text': 'Apa obat flu di apotek?',
                                    'domain': 'kesehatan',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta pilihan obat flu yang tersedia di apotek; cakupan '
                                              'ketersediaan obat menjadi batas kebutuhan informasi.'},
 'query_c0d491c245b4eb74b4831f06': {'query_text': 'Obat flu apa yang aman untuk penderita jantung?',
                                    'domain': 'kesehatan',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta informasi keamanan obat flu untuk penderita penyakit '
                                              'jantung; kondisi rinci dapat dibahas dalam jawaban.'},
 'query_059dccb382374f9ea0a6e0e4': {'query_text': 'Ibuprofen adalah obat untuk sakit apa?',
                                    'domain': 'kesehatan',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta penjelasan kegunaan ibuprofen; objek obat dan intent '
                                              'informasi jelas.'},
 'query_93992bb0716082c31d631992': {'query_text': 'Apa beda paracetamol dan ibuprofen?',
                                    'domain': 'kesehatan',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta perbandingan paracetamol dan ibuprofen; dua objek '
                                              'pembanding disebutkan.'},
 'query_19e69d08540c749c7e44a20b': {'query_text': 'Apa merk ibuprofen yang aman untuk bayi?',
                                    'domain': 'kesehatan',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta informasi merek dan keamanan ibuprofen untuk bayi; '
                                              'penerimaan query tidak membenarkan premis bahwa obat tersebut '
                                              'selalu aman.'},
 'query_8db57e0ed2f01f4d7f83d167': {'query_text': 'Ibuprofen apakah aman untuk ibu menyusui?',
                                    'domain': 'kesehatan',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta informasi keamanan ibuprofen selama menyusui; '
                                              'kelompok pengguna disebutkan.'},
 'query_aa7b20b65257a6f3c6557db8': {'query_text': 'Apa obat sariawan biar cepat sembuh?',
                                    'domain': 'kesehatan',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta pilihan obat untuk membantu pemulihan sariawan; '
                                              'kebutuhan informasi kesehatan jelas.'},
 'query_55407eea809802435d023af0': {'query_text': 'Apakah you c 1000 bisa menyembuhkan sariawan?',
                                    'domain': 'kesehatan',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta evaluasi klaim produk you c 1000 terhadap sariawan; '
                                              'premis khasiat boleh dikoreksi dalam jawaban.'},
 'query_28259a0b89c11333bb0903bd': {'query_text': 'Apa obat sariawan yang aman untuk ibu menyusui?',
                                    'domain': 'kesehatan',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta informasi pilihan obat sariawan pada ibu menyusui; '
                                              'kondisi pengguna membedakan query dari pertanyaan umum.'},
 'query_b2ab4ffe14ddc19904ac8fdc': {'query_text': 'Apa saja obat sariawan yang aman untuk ibu hamil muda?',
                                    'domain': 'kesehatan',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta informasi keamanan obat sariawan pada awal kehamilan; '
                                              'kelompok pengguna dan kondisinya eksplisit.'},
 'query_d5746db2136d23b26de828d7': {'query_text': '5 Langkah Menghentikan mencret?',
                                    'domain': 'kesehatan',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta lima langkah penanganan diare; mencret dapat dipahami '
                                              'sebagai istilah sehari-hari dan angka lima adalah format '
                                              'permintaan.'},
 'query_630c251871e25191647fd8b2': {'query_text': 'Apa obat untuk diare yang manjur?',
                                    'domain': 'kesehatan',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta informasi pilihan obat diare; kata manjur tidak '
                                              'dianggap jaminan keberhasilan obat.'},
 'query_dba79dc2ab5b6289aaa83c23': {'query_text': 'Minum apa agar tidak BAB cair?',
                                    'domain': 'kesehatan',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta informasi minuman untuk menangani BAB cair; objek '
                                              'minuman membedakan kebutuhan informasi dari daftar obat '
                                              'diare.'},
 'query_214008bd53b6e4ede63170d9': {'query_text': 'Apa saja obat diare alami yang aman untuk ibu hamil?',
                                    'domain': 'kesehatan',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta informasi penanganan diare dengan bahan alami pada '
                                              'kehamilan; alami dan aman merupakan premis yang boleh '
                                              'dievaluasi.'},
 'query_97762fcaf2fe972d0fc7d6a5': {'query_text': 'Apa ciri-ciri asam lambung parah?',
                                    'domain': 'kesehatan',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta tanda atau gejala kondisi asam lambung yang berat; '
                                              'tujuan informasi kesehatan jelas.'},
 'query_d956d6dd21cd29c5fd5b22bd': {'query_text': 'Minum apa biar lambung cepat sembuh?',
                                    'domain': 'kesehatan',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'excluded',
                                    'reason': 'Kondisi lambung yang hendak disembuhkan tidak disebutkan; '
                                              'menyimpulkan asam lambung, luka, atau keluhan lain '
                                              'membutuhkan keyword asal.'},
 'query_5e39bcdd9d48ff0e704ab394': {'query_text': 'Bagaimana cara mengobati asam lambung dengan cepat?',
                                    'domain': 'kesehatan',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta cara menangani keluhan asam lambung; kecepatan '
                                              'pemulihan merupakan harapan pengguna, bukan fakta yang '
                                              'diasumsikan.'},
 'query_398c32d5232039231fc629b0': {'query_text': 'Apakah obat lambung aman untuk ginjal?',
                                    'domain': 'kesehatan',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta informasi hubungan keamanan obat lambung dengan '
                                              'ginjal secara umum; jenis obat dapat dibedakan dalam '
                                              'jawaban.'},
 'query_645bc0e5c95417abb01e0e15': {'query_text': 'Obat asam urat yang paling bagus apa ya?',
                                    'domain': 'kesehatan',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta informasi perbandingan obat asam urat; tidak '
                                              'mengasumsikan satu obat terbaik untuk semua kondisi.'},
 'query_6c83d2a2ca79e02640991549': {'query_text': 'Apa ciri-ciri asam urat tinggi?',
                                    'domain': 'kesehatan',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta tanda asam urat tinggi; objek kondisi dan jenis '
                                              'informasi jelas.'},
 'query_2d850ec311779f191d92cb61': {'query_text': 'Apakah ibu hamil boleh minum allopurinol?',
                                    'domain': 'kesehatan',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta informasi keamanan allopurinol pada ibu hamil; obat '
                                              'dan kelompok pengguna eksplisit.'},
 'query_d689bc14a4aeb9589b6b84e3': {'query_text': 'Asam urat dilarang makan apa?',
                                    'domain': 'kesehatan',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta informasi pantangan makanan terkait asam urat; premis '
                                              'larangan dapat diperjelas atau dikoreksi dalam jawaban.'},
 'query_7c10de990fd2d1591dbea9da': {'query_text': 'Apa arti singkatan BCA?',
                                    'domain': 'keuangan',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta kepanjangan BCA dalam penggunaan perbankan Indonesia '
                                              'yang umum; relevan sebagai informasi identitas lembaga '
                                              'keuangan.'},
 'query_c7151f51b56f1104c36fd3ce': {'query_text': 'Tukar uang di BCA apakah bisa?',
                                    'domain': 'keuangan',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta informasi ketersediaan layanan penukaran uang di BCA; '
                                              'jenis penukaran dapat dijelaskan sebagai pilihan layanan.'},
 'query_154e0115605e7ce41734d72b': {'query_text': 'Apakah layanan BCA 1500888 24 jam?',
                                    'domain': 'keuangan',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta informasi jam layanan nomor BCA 1500888; objek '
                                              'layanan dan durasi yang ditanyakan spesifik.'},
 'query_f2f31507dba006980adb3d46': {'query_text': 'Apakah bank BCA tutup hari Sabtu?',
                                    'domain': 'keuangan',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta informasi operasional bank BCA pada hari Sabtu; '
                                              'variasi cabang dapat dijelaskan tanpa mengubah intent.'},
 'query_58e3a23f771e540bd0d94da9': {'query_text': 'Bank BNI singkatan dari apa?',
                                    'domain': 'keuangan',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta kepanjangan nama bank BNI; objek lembaga keuangan '
                                              'eksplisit.'},
 'query_dbb88cb70b3ea945445ac8e1': {'query_text': 'CS BNI apakah 24 jam?',
                                    'domain': 'keuangan',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta informasi jam layanan customer service BNI; CS '
                                              'merupakan istilah layanan yang dapat dipahami dari konteks '
                                              'bank.'},
 'query_87fd5674e4ba2c65107acfed': {'query_text': 'Buka rekening BNI bayar berapa?',
                                    'domain': 'keuangan',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta informasi biaya atau setoran awal membuka rekening '
                                              'BNI; variasi produk dapat dijelaskan dalam jawaban.'},
 'query_833291f4f317affa759185d5': {'query_text': 'Apakah BNI ada pinjaman KUR?',
                                    'domain': 'keuangan',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta informasi ketersediaan produk KUR di BNI; objek '
                                              'produk dan bank jelas.'},
 'query_8a38831db342b8d337357419': {'query_text': 'Bank BTN itu bank apa?',
                                    'domain': 'keuangan',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta penjelasan identitas dan jenis bank BTN; cakupannya '
                                              'lebih luas dari sekadar kepanjangan nama.'},
 'query_9ef174370f4c1b9fd47cc489': {'query_text': 'Apa kepanjangan dari BTN?',
                                    'domain': 'keuangan',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta kepanjangan BTN dalam penggunaan nama bank Indonesia '
                                              'yang umum; kebutuhan informasi identitas lembaga keuangan '
                                              'jelas.'},
 'query_088a837b0e52a69bd7bb8ab4': {'query_text': 'Minimal saldo di BTN berapa?',
                                    'domain': 'keuangan',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta informasi saldo minimum rekening BTN; jenis tabungan '
                                              'yang belum disebutkan dapat dijelaskan per produk.'},
 'query_d38d35df37d7cb9e2314112d': {'query_text': 'Pinjaman apa saja di bank BTN?',
                                    'domain': 'keuangan',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta daftar produk pinjaman bank BTN; intent informasional '
                                              'keuangan jelas.'},
 'query_6c4bf78a038c446ee01e3f43': {'query_text': 'Cara cek BPJS apakah masih aktif atau tidak?',
                                    'domain': 'keuangan',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'excluded',
                                    'reason': 'BPJS tidak dibedakan antara Kesehatan dan Ketenagakerjaan; '
                                              'memilih prosedur pemeriksaan status memerlukan penentuan '
                                              'program dari luar teks.'},
 'query_c6e74d9936ec742750bc73e2': {'query_text': 'BPJS kelas 1, 2, 3 bayar berapa?',
                                    'domain': 'keuangan',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta informasi iuran BPJS menurut kelas 1, 2, dan 3; '
                                              'penanda kelas memperjelas konteks pembiayaan jaminan '
                                              'kesehatan.'},
 'query_0dfffebef587973757ec0e57': {'query_text': 'Bagaimana cara mendaftar BPJS Kesehatan gratis?',
                                    'domain': 'keuangan',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta prosedur pendaftaran BPJS Kesehatan dengan bantuan '
                                              'pembiayaan; kata gratis merupakan premis yang perlu '
                                              'dijelaskan syaratnya.'},
 'query_0ac6905af807682bbb8a576c': {'query_text': 'Bagaimana cara cek BPJS lewat WA?',
                                    'domain': 'keuangan',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'excluded',
                                    'reason': 'Objek yang hendak dicek tidak disebutkan: status, tagihan, '
                                              'saldo, atau kepesertaan; jenis BPJS juga tidak ditentukan.'},
 'query_57ed938f0acb40784baa3d42': {'query_text': 'Apakah harga BBM per 1 April 2026 naik?',
                                    'domain': 'keuangan',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta pemeriksaan perubahan harga BBM pada tanggal '
                                              'tertentu; premis kenaikan tidak dianggap benar sebelum '
                                              'dijawab.'},
 'query_e43110922e991876843c1e5c': {'query_text': 'Harga pertamax hari ini berapa?',
                                    'domain': 'keuangan',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta informasi harga Pertamax saat pengumpulan; produk '
                                              'eksplisit dan harga dapat dijelaskan menurut wilayah.'},
 'query_3848b57c77a81700a63afb6f': {'query_text': '1 liter pertamax berapa?',
                                    'domain': 'keuangan',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'excluded',
                                    'duplicate_of': 'query_e43110922e991876843c1e5c',
                                    'reason': 'Dikeluarkan sebagai variasi kebutuhan informasi yang sama '
                                              "dengan 'Harga pertamax hari ini berapa?'. Gunakan query "
                                              'tersebut sebagai wakil; teks sumber tetap disimpan.'},
 'query_e5cde6f39a1bf3a1d090ce92': {'query_text': 'Berapa harga pertamax per 1 April 2026?',
                                    'domain': 'keuangan',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta harga Pertamax pada 1 April 2026; tanggal historis '
                                              'membedakan kebutuhan informasi dari harga hari ini.'},
 'query_2420b9d092cb357fa25654d5': {'query_text': 'Berapa harga 1 gram emas perhiasan hari ini?',
                                    'domain': 'keuangan',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta harga perhiasan emas satu gram hari ini; jenis produk '
                                              'membedakan dari harga emas tanpa kategori produk.'},
 'query_6ee1458a9e019f676561dbf6': {'query_text': 'Berapa harga 1 gram emas hari ini?',
                                    'domain': 'keuangan',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta informasi harga satu gram emas hari ini; kadar, '
                                              'merek, dan produk dapat dibedakan dalam jawaban.'},
 'query_80b063d9a0d3099c9c9a61be': {'query_text': 'Harga 1 gram cincin emas berapa?',
                                    'domain': 'keuangan',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta kisaran harga cincin emas satu gram; bentuk produk '
                                              'cincin memberi batas informasi tersendiri.'},
 'query_1394d336b44a4830a21176a6': {'query_text': 'Harga kalung emas 24 karat 1 gramnya berapa?',
                                    'domain': 'keuangan',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta harga kalung emas 24 karat dengan berat satu gram; '
                                              'bentuk, kadar, dan berat produk eksplisit.'},
 'query_2bcdcd5ecd249998dd86dbf4': {'query_text': 'Apakah WhatsApp aplikasi?',
                                    'domain': 'teknologi',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta penjelasan apakah WhatsApp termasuk aplikasi; '
                                              'pertanyaan sederhana tetap informasional dalam domain '
                                              'teknologi.'},
 'query_f331d85809d860016f384076': {'query_text': 'Ada apa dengan WhatsApp hari ini?',
                                    'domain': 'teknologi',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta informasi perkembangan atau gangguan terkini '
                                              'WhatsApp; jawaban tidak harus mengasumsikan ada gangguan hari '
                                              'ini.'},
 'query_5132de567f247f49ff5adfc7': {'query_text': 'Apa WA?',
                                    'domain': 'teknologi',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta pengertian WA dalam penggunaan singkatan WhatsApp '
                                              'yang umum; intent definisi jelas tanpa rujukan personal.'},
 'query_8e1566085fcccc3ba7ddee98': {'query_text': 'Android 5 apakah bisa WA?',
                                    'domain': 'teknologi',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta informasi kompatibilitas WhatsApp dengan Android 5; '
                                              'versi sistem dan aplikasi disebutkan.'},
 'query_c26a81d707f33fa8b09b8222': {'query_text': 'Login email gimana?',
                                    'domain': 'teknologi',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta prosedur login email secara umum; layanan yang tidak '
                                              'disebutkan dapat dijelaskan sebagai beberapa pilihan.'},
 'query_87756c4ecbc412616777f4be': {'query_text': 'Gmail itu yang mana?',
                                    'domain': 'teknologi',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'excluded',
                                    'reason': 'Frasa yang mana meminta identifikasi di antara pilihan yang '
                                              'tidak disediakan; tidak jelas apakah merujuk aplikasi, '
                                              'alamat, atau akun.'},
 'query_273984a3a8b7790f9f142819': {'query_text': 'Apa itu Inbox Gmail?',
                                    'domain': 'teknologi',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta definisi Inbox Gmail; fitur dan layanan yang dimaksud '
                                              'jelas.'},
 'query_53968339ec60eb65fbebc252': {'query_text': 'Google apa email saya?',
                                    'domain': 'teknologi',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'excluded',
                                    'reason': 'Meminta identifikasi alamat email pribadi pengguna tanpa '
                                              'konteks akun; bukan kebutuhan informasi artikel yang dapat '
                                              'dijawab dari sumber umum.'},
 'query_a34e6fca4e0d826c3822dcfb': {'query_text': 'Program Accurate itu apa?',
                                    'domain': 'teknologi',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta pengertian program Accurate; kata program memperjelas '
                                              'bahwa objeknya perangkat lunak.'},
 'query_ee58f50b3340a73e5dbd67d3': {'query_text': 'Apa arti Accurate?',
                                    'domain': 'teknologi',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'excluded',
                                    'reason': 'Meminta arti kata Accurate tanpa penanda perangkat lunak; '
                                              'interpretasi sebagai aplikasi teknologi memerlukan keyword '
                                              'atau konteks tambahan.'},
 'query_8e0624159e670097c2d4e52f': {'query_text': 'Apa itu myob dan Accurate?',
                                    'domain': 'teknologi',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta penjelasan MYOB dan Accurate secara bersama; pasangan '
                                              'nama tersebut memperjelas konteks perangkat lunak akuntansi.'},
 'query_c45426225cdd5d671e0c644e': {'query_text': 'Accurate kerja apa?',
                                    'domain': 'teknologi',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'excluded',
                                    'reason': 'Frasa Accurate kerja apa tidak membedakan fungsi perangkat '
                                              'lunak dari pekerjaan atau organisasi; membutuhkan penambahan '
                                              'konteks.'},
 'query_0c91974960acd576ef150d04': {'query_text': 'Agar campak keluar semua minum apa?',
                                    'domain': 'kesehatan',
                                    'previous_status': 'needs_review',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta informasi minuman terkait campak dengan premis campak '
                                              'harus keluar semua; premis dipertahankan sebagai pertanyaan '
                                              'yang boleh dikoreksi.'},
 'query_6decec2d9f07bef4218a002f': {'query_text': '1 juta dipotong pajak jadi berapa?',
                                    'domain': 'keuangan',
                                    'previous_status': 'needs_review',
                                    'language': 'id',
                                    'status': 'excluded',
                                    'reason': 'Jumlah satu juta tidak dikaitkan dengan transaksi atau objek '
                                              'pajak tertentu; menafsirkannya sebagai gaji, belanja, hadiah, '
                                              'atau lainnya memerlukan tebakan.'},
 'query_2bd708f39c0684bdf470fb32': {'query_text': 'Sunscreen yang bagus dan aman merk apa?',
                                    'domain': 'kesehatan',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta rekomendasi merek sunscreen dengan pertimbangan '
                                              'keamanan; aspek keamanan eksplisit membedakan dari daftar '
                                              'merek umum.'},
 'query_c26f12fa117298aec815ef71': {'query_text': 'Ibu hamil sebaiknya menggunakan sunscreen apa?',
                                    'domain': 'kesehatan',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'excluded',
                                    'duplicate_of': 'query_f85473ab3176c1d5731d04db',
                                    'reason': 'Dikeluarkan sebagai variasi kebutuhan informasi yang sama '
                                              "dengan 'Sunscreen apa yang cocok untuk ibu hamil?'. Gunakan "
                                              'query tersebut sebagai wakil; teks sumber tetap disimpan.'},
 'query_bc3928ea1c48b7d3b4ef10ef': {'query_text': 'Apa sunscreen nomor 1 di Indonesia?',
                                    'domain': 'kesehatan',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta perbandingan sunscreen di Indonesia; nomor satu '
                                              'diperlakukan sebagai permintaan evaluasi, bukan pengakuan '
                                              'peringkat resmi universal.'},
 'query_20087a600ac1e599f2885a6a': {'query_text': 'Apa merk sunscreen yang bagus dan harganya?',
                                    'domain': 'kesehatan',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta rekomendasi merek sunscreen beserta harga; informasi '
                                              'harga menambah kebutuhan dibanding daftar merek saja.'},
 'query_4b75e7beca88f3be7f7b6bc1': {'query_text': 'Laptop asus ram 4GB apakah bagus?',
                                    'domain': 'teknologi',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta evaluasi kecukupan laptop ASUS RAM 4 GB; model dan '
                                              'kebutuhan penggunaan dapat dijelaskan sebagai faktor '
                                              'penilaian.'},
 'query_70f1df3d2a47960a488d1bc1': {'query_text': 'Laptop RAM 4GB harga berapa?',
                                    'domain': 'teknologi',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta kisaran harga laptop RAM 4 GB tanpa membatasi merek; '
                                              'cakupan berbeda dari pertanyaan khusus ASUS.'},
 'query_9e4a0aaa08512c525c908ee6': {'query_text': 'Berapa harga laptop bekas Asus dengan RAM 4GB?',
                                    'domain': 'teknologi',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta harga laptop ASUS RAM 4 GB bekas; kondisi bekas '
                                              'membedakan dari pencarian harga umum atau baru.'},
 'query_169786ad40e908636b010d5b': {'query_text': 'Berapa harga laptop Asus dengan RAM 4GB dan Windows 10?',
                                    'domain': 'teknologi',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta harga laptop ASUS RAM 4 GB dengan Windows 10; batas '
                                              'sistem operasi disebutkan secara eksplisit.'},
 'query_21382bfd2bee3ee4ed3df7bc': {'query_text': 'Bagian apa yang harus dipijat saat batuk?',
                                    'domain': 'kesehatan',
                                    'previous_status': 'needs_review',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta informasi lokasi pijat dalam kaitannya dengan batuk; '
                                              'kata harus adalah premis pengguna yang boleh dikoreksi, bukan '
                                              'rekomendasi peneliti.'},
 'query_cbf5f652767f04819a53c70f': {'query_text': 'Laptop harga 3 jutaan merk apa?',
                                    'domain': 'teknologi',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta merek atau pilihan laptop pada anggaran tiga juta; '
                                              'batas harga membuat kebutuhan informasi jelas.'},
 'query_12a85f1e7c163dbbbdd2c454': {'query_text': 'Laptop asus ram 8 harga berapa?',
                                    'domain': 'teknologi',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta harga laptop ASUS RAM 8 GB; variasi model dapat '
                                              'dijelaskan sebagai kisaran.'},
 'query_5897c4ecc885b5769ed62bea': {'query_text': 'Laptop asus ram 16 harga berapa?',
                                    'domain': 'teknologi',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'excluded',
                                    'duplicate_of': 'query_663d7ce44565ad0252c3afba',
                                    'reason': 'Dikeluarkan sebagai variasi kebutuhan informasi yang sama '
                                              "dengan 'Berapa harga laptop ASUS 16 ram?'. Gunakan query "
                                              'tersebut sebagai wakil; teks sumber tetap disimpan.'},
 'query_688cda6227e31b0179e1882a': {'query_text': 'Gaji 3 juta bayar NPWP berapa?',
                                    'domain': 'keuangan',
                                    'previous_status': 'needs_review',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta penjelasan hubungan gaji tiga juta dengan pembayaran '
                                              'terkait NPWP; kekeliruan penyebutan NPWP sebagai pajak adalah '
                                              'premis yang boleh diperjelas.'},
 'query_76ef297c5d169bcc69c44e1e': {'query_text': 'vitamin C untuk ibu hamil merk apa?',
                                    'domain': 'kesehatan',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'excluded',
                                    'duplicate_of': 'query_5b419e03fa87251a57e26881',
                                    'reason': 'Dikeluarkan sebagai variasi kebutuhan informasi yang sama '
                                              "dengan 'Apa saja merk vitamin C yang aman untuk ibu hamil?'. "
                                              'Gunakan query tersebut sebagai wakil; teks sumber tetap '
                                              'disimpan.'},
 'query_69a3b474d859bfa02f4e0d3f': {'query_text': 'Apakah vitamin C boleh untuk penderita diabetes?',
                                    'domain': 'kesehatan',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta informasi kelayakan vitamin C untuk penderita '
                                              'diabetes; kelompok pengguna eksplisit dan klaim keamanan '
                                              'belum diasumsikan.'},
 'query_0c65c58e2b389e269d54ec04': {'query_text': 'Apakah vitamin C cocok untuk batuk?',
                                    'domain': 'kesehatan',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta evaluasi penggunaan vitamin C dalam kaitannya dengan '
                                              'batuk; pertanyaan tidak dianggap bukti manfaat.'},
 'query_cda95c12405ad80b0d6ffe34': {'query_text': 'Ibu hamil migrain obatnya apa?',
                                    'domain': 'kesehatan',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta informasi obat migrain saat hamil; objek kondisi dan '
                                              'kelompok pengguna jelas.'},
 'query_547eb82de08affe79866d43b': {'query_text': 'Bagaimana cara pinjol melacak lokasi?',
                                    'domain': 'keuangan',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta penjelasan mekanisme pelacakan lokasi oleh pinjol; '
                                              'berbeda dari sekadar menanyakan apakah pelacakan mungkin '
                                              'terjadi.'},
 'query_c2d6cb2d8d127eb53a4bf826': {'query_text': 'Apa yang akan terjadi jika kita tidak membayar pinjol?',
                                    'domain': 'keuangan',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta penjelasan konsekuensi tidak membayar pinjaman '
                                              'online; intent keuangan jelas.'},
 'query_f4189f4ae98efe243b42ac1a': {'query_text': 'Kenapa DC bisa tahu lokasi kita?',
                                    'domain': 'keuangan',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'excluded',
                                    'reason': 'Singkatan DC tidak dijelaskan dan dapat merujuk beberapa '
                                              'objek; menganggapnya penagih utang membutuhkan konteks pinjol '
                                              'dari sumber asal.'},
 'query_7f1c282e03d0c463855a3f31': {'query_text': 'Apakah pinjol bisa melacak kontak kita?',
                                    'domain': 'keuangan',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta informasi kemampuan pinjol mengakses atau melacak '
                                              'kontak; kontak merupakan objek berbeda dari lokasi.'},
 'query_e924cb8aade93f9e1b3ad966': {'query_text': 'Sunscreen spf 50 pa+++ untuk kulit apa?',
                                    'domain': 'kesehatan',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta kesesuaian sunscreen SPF 50 PA+++ untuk jenis kulit; '
                                              'tingkat PA dan tujuan pemakaian eksplisit.'},
 'query_a3458320ff07285d9c59ac4c': {'query_text': 'Sunscreen SPF 50 yang paling bagus merk apa?',
                                    'domain': 'kesehatan',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta perbandingan merek sunscreen SPF 50; kriteria terbaik '
                                              'dapat dijelaskan tanpa menetapkan pemenang universal.'},
 'query_70eb659010935e6435b666c2': {'query_text': 'Sunscreen merk apa yang aman untuk ibu hamil?',
                                    'domain': 'kesehatan',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'excluded',
                                    'duplicate_of': 'query_f85473ab3176c1d5731d04db',
                                    'reason': 'Dikeluarkan sebagai variasi kebutuhan informasi yang sama '
                                              "dengan 'Sunscreen apa yang cocok untuk ibu hamil?'. Gunakan "
                                              'query tersebut sebagai wakil; teks sumber tetap disimpan.'},
 'query_2ee7435f146c724183a71a1a': {'query_text': 'Sunscreen apa yang SPF nya 50?',
                                    'domain': 'kesehatan',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta daftar sunscreen SPF 50 tanpa membatasi tingkat PA; '
                                              'cakupan berbeda dari daftar SPF 50 PA++++.'},
 'query_cd9da732ec23f97f9ceb5b5d': {'query_text': 'Laptop RAM 16GB harga berapa?',
                                    'domain': 'teknologi',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta harga laptop RAM 16 GB tanpa membatasi merek; cakupan '
                                              'berbeda dari pertanyaan khusus ASUS.'},
 'query_36de90865c59975945251688': {'query_text': 'Berapa harga laptop ASUS dengan Core i7 dan RAM 16GB '
                                                  'baru?',
                                    'domain': 'teknologi',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta harga laptop ASUS baru dengan Core i7 dan RAM 16 GB; '
                                              'prosesor dan kondisi barang membatasi kebutuhan informasi.'},
 'query_1e4d38a9967cbd24350b7122': {'query_text': 'Laptop RAM 16GB merk apa?',
                                    'domain': 'teknologi',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta pilihan merek laptop RAM 16 GB; kebutuhan merek '
                                              'berbeda dari harga.'},
 'query_59212d8f914e4946d4269586': {'query_text': 'Ram 16GB harganya berapa?',
                                    'domain': 'teknologi',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta harga komponen RAM kapasitas 16 GB; jenis dan '
                                              'generasi RAM dapat dijelaskan sebagai faktor kisaran harga.'},
 'query_95f0b0b53388186ecb897937': {'query_text': 'Bolehkah ibu hamil minum vitamin C?',
                                    'domain': 'kesehatan',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta informasi keamanan minum vitamin C pada kehamilan; '
                                              'berbeda dari mencari merek suplemen tertentu.'},
 'query_87a9d099070d8ca5800940c0': {'query_text': 'Apakah penderita gagal ginjal boleh minum vitamin C?',
                                    'domain': 'kesehatan',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta informasi kelayakan vitamin C pada penderita gagal '
                                              'ginjal; kelompok pengguna membedakan query dari kondisi '
                                              'kesehatan lain.'},
 'query_047d79f269c9728633724674': {'query_text': 'Apa saja laptop bagus dengan harga 2 jutaan?',
                                    'domain': 'teknologi',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'excluded',
                                    'duplicate_of': 'query_8c3d0d988399c48b0f79fd03',
                                    'reason': 'Dikeluarkan sebagai variasi kebutuhan informasi yang sama '
                                              "dengan 'Laptop harga 2 jutaan merk apa?'. Gunakan query "
                                              'tersebut sebagai wakil; teks sumber tetap disimpan.'},
 'query_2226d1c68e585bb83dab91e9': {'query_text': 'Laptop bagus dan murah merk apa?',
                                    'domain': 'teknologi',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta rekomendasi laptop dengan pertimbangan mutu dan harga '
                                              'murah secara umum; tidak membatasi ke anggaran dua atau tiga '
                                              'juta.'},
 'query_53c7f0b499e600953c1c4b6e': {'query_text': 'Apa laptop 2 jutaan yang cocok untuk kuliah?',
                                    'domain': 'teknologi',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta rekomendasi laptop untuk kuliah pada anggaran dua '
                                              'juta; kebutuhan penggunaan menambah konteks dibanding daftar '
                                              'merek umum.'},
 'query_79f318806e1b9d6fa595e07f': {'query_text': 'Berapa harga laptop Acer dengan harga 2 juta?',
                                    'domain': 'teknologi',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'excluded',
                                    'reason': 'Harga yang ditanyakan sudah dinyatakan sebagai dua juta; teks '
                                              'melingkar dan perlu ditulis ulang menjadi pertanyaan model '
                                              'atau spesifikasi agar kebutuhan informasinya jelas.'},
 'query_6206bc86619a2731c413225e': {'query_text': 'Cara cek NIK KTP apakah terdaftar?',
                                    'domain': 'keuangan',
                                    'previous_status': 'needs_review',
                                    'language': 'id',
                                    'status': 'excluded',
                                    'reason': 'Sistem tempat NIK hendak dicek tidak disebutkan; memilih '
                                              'Dukcapil, NPWP, atau layanan lain memerlukan konteks '
                                              'tambahan.'},
 'query_b77ff467afa445782418b0d5': {'query_text': 'Apa VPN yang paling aman?',
                                    'domain': 'teknologi',
                                    'previous_status': 'needs_review',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta perbandingan keamanan VPN secara umum; berbeda dari '
                                              'daftar VPN gratis karena tidak membatasi skema biaya.'},
 'query_f01b137252df86be12c90169': {'query_text': 'VPN apa yang bebas blokir?',
                                    'domain': 'teknologi',
                                    'previous_status': 'needs_review',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta informasi pilihan VPN terkait akses internet yang '
                                              'diblokir; jenis blokir dan keterbatasan layanan dapat '
                                              'dijelaskan tanpa menjamin bebas blokir.'},
 'query_c86212c712a3d249a80b4894': {'query_text': 'Pinjol yang paling aman apa?',
                                    'domain': 'keuangan',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'excluded',
                                    'duplicate_of': 'query_b458d3df1505cf3e2f5069a9',
                                    'reason': 'Dikeluarkan sebagai variasi kebutuhan informasi yang sama '
                                              "dengan 'Pinjaman online aman dimana?'. Gunakan query tersebut "
                                              'sebagai wakil; teks sumber tetap disimpan.'},
 'query_7c900b5e385352738af619ed': {'query_text': 'Di mana tempat pinjam uang online yang terpercaya?',
                                    'domain': 'keuangan',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'excluded',
                                    'duplicate_of': 'query_b458d3df1505cf3e2f5069a9',
                                    'reason': 'Dikeluarkan sebagai variasi kebutuhan informasi yang sama '
                                              "dengan 'Pinjaman online aman dimana?'. Gunakan query tersebut "
                                              'sebagai wakil; teks sumber tetap disimpan.'},
 'query_0a0cc275e062b690f9db485b': {'query_text': 'Pinjam uang yang aman di aplikasi apa?',
                                    'domain': 'keuangan',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'excluded',
                                    'duplicate_of': 'query_b458d3df1505cf3e2f5069a9',
                                    'reason': 'Dikeluarkan sebagai variasi kebutuhan informasi yang sama '
                                              "dengan 'Pinjaman online aman dimana?'. Gunakan query tersebut "
                                              'sebagai wakil; teks sumber tetap disimpan.'},
 'query_5ae34a6ed5c9c1a1bfff6c58': {'query_text': 'Pinjaman online yang resmi apa saja?',
                                    'domain': 'keuangan',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta daftar pinjaman online berstatus resmi; status resmi '
                                              'menjadi kriteria eksplisit yang perlu diverifikasi dalam '
                                              'jawaban.'},
 'query_ad07e8eabd05e6d61115398a': {'query_text': 'Apa saja ciri-ciri pinjaman online penipu?',
                                    'domain': 'keuangan',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta ciri-ciri penipuan pinjaman online; fokus pengenalan '
                                              'penipuan berbeda dari daftar layanan atau risiko umum.'},
 'query_096f15aa95b80b75c00a26ad': {'query_text': 'Apa yang dimaksud dengan pinjaman daring?',
                                    'domain': 'keuangan',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta definisi pinjaman daring; berbeda dari daftar produk '
                                              'dan penilaian keamanannya.'},
 'query_20d2bdf3c784a2204e62b6a4': {'query_text': 'Apa resiko menggunakan pinjaman online?',
                                    'domain': 'keuangan',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta rincian risiko penggunaan pinjaman online; kebutuhan '
                                              'daftar risiko dibedakan dari pertanyaan keamanan umum.'},
 'query_a691dc05862939f4fc8271e2': {'query_text': 'Sayuran vitamin C apa saja?',
                                    'domain': 'kesehatan',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta sumber vitamin C dari kelompok sayuran; cakupan '
                                              'makanan lebih spesifik dari daftar makanan umum.'},
 'query_8f8b96b5ee9d19982e3288ed': {'query_text': 'Kurang vitamin C harus makan apa?',
                                    'domain': 'kesehatan',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta pilihan makanan pada kondisi kekurangan vitamin C; '
                                              'kondisi defisiensi menambah kebutuhan informasi dibanding '
                                              'daftar sumber vitamin umum.'},
 'query_ec4d2d949954b19e4ff34785': {'query_text': 'vitamin C tertinggi di buah apa?',
                                    'domain': 'kesehatan',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta perbandingan kandungan vitamin C antarbuah; fokus '
                                              'perbandingan kadar berbeda dari daftar sumber umum.'},
 'query_3ab88cb5dc5ddb76b5e21cf6': {'query_text': 'Apa saja 10 makanan yang mengandung vitamin C?',
                                    'domain': 'kesehatan',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'excluded',
                                    'duplicate_of': 'query_ca4c95a4f470d2d45c0dba75',
                                    'reason': 'Dikeluarkan sebagai variasi kebutuhan informasi yang sama '
                                              "dengan 'vitamin C ada di makanan apa?'. Gunakan query "
                                              'tersebut sebagai wakil; teks sumber tetap disimpan.'},
 'query_ae56444b7fd4b25020e9bb49': {'query_text': 'Berapa harga emas 1 gram hari ini?',
                                    'domain': 'keuangan',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'excluded',
                                    'duplicate_of': 'query_6ee1458a9e019f676561dbf6',
                                    'reason': 'Dikeluarkan sebagai variasi kebutuhan informasi yang sama '
                                              "dengan 'Berapa harga 1 gram emas hari ini?'. Gunakan query "
                                              'tersebut sebagai wakil; teks sumber tetap disimpan.'},
 'query_b9e3eeba6806e4664a1b4ce9': {'query_text': 'Berapa harga menggadaikan emas 1 gram di Pegadaian?',
                                    'domain': 'keuangan',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta nilai atau biaya gadai emas satu gram di Pegadaian; '
                                              'konteks gadai membedakan dari pembelian emas.'},
 'query_903abcb4e23316d8e9073333': {'query_text': 'Berapa emas 1 mayam?',
                                    'domain': 'keuangan',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'excluded',
                                    'reason': 'Besaran yang diminta untuk satu mayam tidak disebutkan: '
                                              'berat, harga, atau nilai lain; menetapkannya membutuhkan '
                                              'tebakan.'},
 'query_e05519d62d9d2b69c515b688': {'query_text': 'Beli emas di Pegadaian minimal berapa?',
                                    'domain': 'keuangan',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta batas minimum pembelian emas di Pegadaian; ketentuan '
                                              'dapat dijelaskan menurut produk dalam berat dan nominal.'},
 'query_ef2797aae12b1e83026b7cf3': {'query_text': 'Pinjam 3 juta di Pegadaian cicilan berapa?',
                                    'domain': 'keuangan',
                                    'previous_status': 'needs_review',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta perhitungan cicilan pinjaman tiga juta di Pegadaian; '
                                              'tenor dan produk dapat dibedakan sebagai skenario.'},
 'query_2075fd71d4f80432d12df002': {'query_text': 'Pinjam 2 juta di Pegadaian angsuran berapa?',
                                    'domain': 'keuangan',
                                    'previous_status': 'needs_review',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta perhitungan angsuran pinjaman dua juta di Pegadaian; '
                                              'nominal membedakan skenario dari pinjaman lain.'},
 'query_265c9cd4e05e5e382e9ececc': {'query_text': 'Pinjaman 1 juta di Pegadaian bunganya berapa?',
                                    'domain': 'keuangan',
                                    'previous_status': 'needs_review',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta informasi bunga pinjaman satu juta di Pegadaian; '
                                              'produk dan jangka waktu menjadi faktor perhitungan.'},
 'query_f8e95c75f0735f5655f53a22': {'query_text': 'Pinjam 10 juta di Pegadaian cicilan berapa?',
                                    'domain': 'keuangan',
                                    'previous_status': 'needs_review',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta perhitungan cicilan pinjaman sepuluh juta di '
                                              'Pegadaian; nominal disebutkan dan tenor dapat dijelaskan '
                                              'sebagai pilihan.'},
 'query_b18cf6dca5107f209782a7f1': {'query_text': 'Apa saja 10 pinjaman online terbaik?',
                                    'domain': 'keuangan',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta perbandingan sepuluh layanan pinjaman online; terbaik '
                                              'diperlakukan sebagai kriteria yang perlu dijelaskan, bukan '
                                              'peringkat resmi yang pasti.'},
 'query_0fa373219a8cf1f849cca9ff': {'query_text': 'Pinjaman online apa yang paling aman?',
                                    'domain': 'keuangan',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'excluded',
                                    'duplicate_of': 'query_b458d3df1505cf3e2f5069a9',
                                    'reason': 'Dikeluarkan sebagai variasi kebutuhan informasi yang sama '
                                              "dengan 'Pinjaman online aman dimana?'. Gunakan query tersebut "
                                              'sebagai wakil; teks sumber tetap disimpan.'},
 'query_1f4a60f45dc135dada159d30': {'query_text': 'Pinjaman online cepat cair apa saja?',
                                    'domain': 'keuangan',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta layanan pinjaman online dengan proses pencairan '
                                              'cepat; kecepatan pencairan merupakan kriteria tersendiri.'},
 'query_c364c23617fee67452a62515': {'query_text': 'Pinjaman daring apa?',
                                    'domain': 'keuangan',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'excluded',
                                    'duplicate_of': 'query_096f15aa95b80b75c00a26ad',
                                    'reason': 'Dikeluarkan sebagai variasi kebutuhan informasi yang sama '
                                              "dengan 'Apa yang dimaksud dengan pinjaman daring?'. Gunakan "
                                              'query tersebut sebagai wakil; teks sumber tetap disimpan.'},
 'query_b592895b7ce3cc2c8304a51a': {'query_text': 'Apakah vitamin C cocok untuk penderita diabetes?',
                                    'domain': 'kesehatan',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'excluded',
                                    'duplicate_of': 'query_69a3b474d859bfa02f4e0d3f',
                                    'reason': 'Dikeluarkan sebagai variasi kebutuhan informasi yang sama '
                                              "dengan 'Apakah vitamin C boleh untuk penderita diabetes?'. "
                                              'Gunakan query tersebut sebagai wakil; teks sumber tetap '
                                              'disimpan.'},
 'query_f61faca1030a3da86420333b': {'query_text': 'Apa manfaat minum vitamin C setiap hari?',
                                    'domain': 'kesehatan',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta evaluasi manfaat konsumsi vitamin C setiap hari; '
                                              'frekuensi rutin memberi konteks berbeda dari fungsi vitamin C '
                                              'secara umum.'},
 'query_856c816acea302a995499bc6': {'query_text': '1 lot saham BRI dapat dividen berapa?',
                                    'domain': 'keuangan',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta perhitungan dividen satu lot saham BRI; periode '
                                              'pembagian yang belum disebutkan dapat dijelaskan dalam '
                                              'jawaban.'},
 'query_6187241da0f877100320eb42': {'query_text': 'Modal 10 juta dapat dividen berapa?',
                                    'domain': 'keuangan',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta perhitungan potensi dividen dari modal sepuluh juta; '
                                              'emiten dan imbal hasil dapat dijelaskan sebagai asumsi, tanpa '
                                              'menjanjikan hasil.'},
 'query_57884d7242f8158f66708bdc': {'query_text': '100 lot BCA dapat dividen berapa?',
                                    'domain': 'keuangan',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta perhitungan dividen seratus lot saham BCA; emiten dan '
                                              'jumlah lot membatasi kebutuhan informasi.'},
 'query_9ce7f2cd10d81fdc7aa37074': {'query_text': 'Apakah dividen bisa dicairkan?',
                                    'domain': 'keuangan',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta penjelasan pencairan dividen; berbeda dari definisi '
                                              'dan perhitungan nominal dividen.'},
 'query_39742a2f02f26c696c5f6a0d': {'query_text': 'Skincare yang aman untuk ibu hamil merk apa saja?',
                                    'domain': 'kesehatan',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta merek skincare dengan pertimbangan keamanan '
                                              'kehamilan; cakupan produk lebih luas dari sunscreen saja.'},
 'query_52a229da1b741acaebd1752c': {'query_text': 'Sunscreen Wardah SPF 50 apakah aman untuk bumil?',
                                    'domain': 'kesehatan',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta evaluasi keamanan sunscreen Wardah SPF 50 pada '
                                              'kehamilan; merek, SPF, dan kelompok pengguna eksplisit.'},
 'query_8ca6a675613c5c479ff0e53f': {'query_text': 'Apakah sunscreen azarine SPF 50 aman untuk ibu hamil?',
                                    'domain': 'kesehatan',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta evaluasi keamanan sunscreen Azarine SPF 50 pada '
                                              'kehamilan; merek membedakan objek dari sunscreen lainnya.'},
 'query_365bef49ee2740f8ce943e89': {'query_text': 'Kenapa bumil tidak boleh pakai chemical sunscreen?',
                                    'domain': 'kesehatan',
                                    'previous_status': 'pending',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta penjelasan dugaan larangan chemical sunscreen saat '
                                              'hamil; premis larangan boleh dikoreksi dan tidak dianggap '
                                              'fakta peneliti.'}}


In [22]:
# Terapkan hanya ke query yang belum selesai ditinjau atau masih milik review asisten ini.
# Keputusan peneliti serta keputusan final lain tidak ditimpa.
REVIEWER = 'asisten_rubrik_v1'
current_rows = read_rows(OUTPUT / 'candidates.csv')
current_by_id = {r['query_id']: r for r in current_rows}
manual_rows = read_rows(ROOT / 'data/manual' / (config['expansion_id'] + '_decisions.csv'))
manual_by_id = {r['query_id']: r for r in manual_rows}
assistant_changes = {}
skipped = []
for query_id, decision in assistant_decisions.items():
    row = current_by_id.get(query_id)
    if row is None or row['query_text'] != decision['query_text'] or row['domain'] != decision['domain']:
        raise ValueError(f'Sumber review berubah atau tidak ditemukan: {query_id}')
    previous = manual_by_id.get(query_id, {})
    unresolved = row['selection_status'] in {'pending', 'needs_review'}
    owned = previous.get('reviewer') == REVIEWER
    if not (unresolved or owned):
        skipped.append(decision['query_text'])
        continue
    change = {k: decision[k] for k in ('language', 'status', 'reason')}
    change['reviewer'] = REVIEWER
    if any(previous.get(k) != v for k, v in change.items()):
        assistant_changes[query_id] = change
if assistant_changes:
    candidates = prepare(config, assistant_changes)
else:
    candidates = current_rows
    print('Tidak ada perubahan keputusan asisten yang perlu disimpan.')
print('Keputusan asisten diperbarui:', len(assistant_changes))
print('Keputusan final lain dipertahankan:', len(skipped))
print('Status kandidat terbaru:', dict(Counter(r['selection_status'] for r in candidates)))
show([current_by_id[qid] | {'review_status': d['status'], 'review_reason': d['reason']}
      for qid, d in assistant_decisions.items()],
     ['query_text', 'domain', 'review_status', 'review_reason'], limit=len(assistant_decisions))


Kemunculan PAA: 324 | query unik dengan domain: 316
Kandidat baru: 309 {'kesehatan': 121, 'keuangan': 109, 'teknologi': 79}
Status: {'accepted': 246, 'excluded': 63}
Output: D:\Kuliah\TA\final-assignment\data\interim\query_expansion\query_expansion_01
Keputusan asisten diperbarui: 149
Keputusan final lain dipertahankan: 0
Status kandidat terbaru: {'accepted': 246, 'excluded': 63}


query_text,domain,review_status,review_reason
Cara mengecek apakah kita dapat bantuan BPJS Ketenagakerjaan?,keuangan,accepted,Meminta prosedur mengecek kelayakan bantuan terkait BPJS Ketenagakerjaan; jenis bantuan dapat dijelaskan sebagai pilihan tanpa mengasumsikan penerima pasti berhak.
BPJS Ketenagakerjaan dicairkan berapa?,keuangan,accepted,"Meminta penjelasan nilai pencairan manfaat BPJS Ketenagakerjaan; program dan kondisi peserta memengaruhi perhitungan, tetapi intent keuangan jelas."
KUR BRI 2026 pinjaman 50 juta angsuran berapa?,keuangan,accepted,Meminta perhitungan angsuran KUR BRI 2026 sebesar 50 juta; tenor yang belum disebutkan dapat disajikan sebagai beberapa skenario.
Siapa investor nomor 1 di Indonesia?,keuangan,accepted,Meminta perbandingan investor terkemuka di Indonesia; kriteria peringkat perlu dijelaskan dalam jawaban dan tidak dianggap ada satu peringkat resmi yang pasti.
Investasi 1 juta per bulan dapat berapa?,keuangan,accepted,"Meminta penjelasan hasil investasi rutin satu juta per bulan; instrumen, durasi, dan imbal hasil menjadi asumsi perhitungan, bukan syarat kejelasan intent."
Bank yang paling aman di Indonesia bank apa?,keuangan,accepted,Meminta perbandingan keamanan bank di Indonesia; jawaban dapat menjelaskan indikator keamanan tanpa menjamin satu bank selalu paling aman.
Pinjam uang 50 juta di bank BCA cicilan berapa?,keuangan,accepted,Meminta perhitungan cicilan pinjaman BCA sebesar 50 juta; produk dan tenor dapat dijelaskan sebagai skenario.
Pinjam 100 juta di Mandiri angsuran berapa?,keuangan,accepted,"Meminta perhitungan angsuran pinjaman Mandiri sebesar 100 juta; produk dan tenor memengaruhi angka, tetapi kebutuhan informasinya jelas."
Driver itu artinya apa?,teknologi,excluded,Objek driver ambigu antara perangkat lunak dan pengemudi; domain teknologi hanya dapat dipastikan dengan konteks tambahan.
Arti drive apa?,teknologi,excluded,Kata drive mempunyai makna umum dan komputasi; teks tidak menentukan objek teknologi sehingga memerlukan tebakan konteks.


Ditampilkan 149 dari 149 baris.


## Tambahan PAA untuk target sekitar 300 query

Pengumpulan 11 September 2026 menggunakan keyword Trends yang belum dicari: cetirizine, rupiah, subsidi tepat, blackbox ai, duckduckgo, gform, uang, maxstream, dan ibox. Total sembilan request pencarian: lima menghasilkan PAA, dua tidak memiliki PAA, dan dua gagal tanpa respons tersimpan. Request gagal tidak diulang otomatis. Kuota terverifikasi berubah dari 179 menjadi 170. Tidak ada panggilan Gemini.

Dari 20 kemunculan PAA, satu query sudah ada: **Tukar uang di BCA apakah bisa?**. Provenance kemunculan tetap disimpan, tetapi query yang sama tidak dihitung dua kali dan keputusan lamanya dipertahankan. Tersisa 19 kandidat baru: **15 accepted dan 4 excluded**, tanpa needs_review/pending baru. Penilaian menggunakan rubrik yang sama dan dicatat sebagai LLM-as-a-judge oleh `asisten_rubrik_target300_v1`; bukan validasi manusia independen.

Total accepted setelah penerapan: **302 query** (129 kesehatan, 103 keuangan, 70 teknologi), terdiri dari 41 query pada main_01 dan 261 query tambahan. Total kandidat ekspansi: 328, terdiri dari 261 accepted dan 67 excluded. Empat penolakan berkaitan dengan pertanyaan rupiah yang tidak lengkap dan form yang tidak eksplisit sebagai objek teknologi.

Keputusan blok ini sudah diterapkan ke CSV secara lokal oleh asisten atas permintaan peneliti. Menjalankan ulang blok tidak menimpa keputusan final peneliti dan tidak mengganti timestamp jika keputusan asisten tidak berubah. Untuk mengoreksi hasil, gunakan bagian Koreksi keputusan oleh peneliti di bawah. Angka ini adalah snapshot sebelum koreksi selanjutnya; daftar query utama main_01 tetap dibekukan.


In [ ]:
# Review tambahan untuk target sekitar 300 query, berdasarkan PAA ekspansi 03 dan 04.
# Teks asli dipertahankan; dictionary berikut menyimpan seluruh keputusan baru.
paa_target_decisions = {'query_ac970b2e40d55143b25c0e27': {'query_text': 'Cetirizine obat untuk penyakit apa?',
                                    'domain': 'kesehatan',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta kegunaan cetirizine; objek obat dan intent penjelasan '
                                              'kesehatan jelas.'},
 'query_6cf188a6d39cc7102864d813': {'query_text': 'Apakah cetirizine aman dikonsumsi oleh ibu menyusui?',
                                    'domain': 'kesehatan',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta informasi keamanan cetirizine selama menyusui; obat '
                                              'dan kelompok pengguna eksplisit. Penerimaan bukan pembenaran '
                                              'klaim aman.'},
 'query_252dd03c875e9fa02f98f675': {'query_text': 'Berapa dosis cetirizine yang aman untuk ibu hamil?',
                                    'domain': 'kesehatan',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta informasi dosis cetirizine pada kehamilan; kebutuhan '
                                              'informasi jelas dan premis keamanan dapat dievaluasi dalam '
                                              'jawaban.'},
 'query_a9e5836a5ec2cc03f68cb7b5': {'query_text': 'Apakah salbutamol bisa diminum bersamaan dengan '
                                                  'cetirizine?',
                                    'domain': 'kesehatan',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta informasi penggunaan bersamaan dua obat yang '
                                              'disebutkan; intent interaksi atau kompatibilitas obat jelas.'},
 'query_5c19718e6e2b0a39bc0cd266': {'query_text': 'Kenapa rupiah?',
                                    'domain': 'keuangan',
                                    'language': 'id',
                                    'status': 'excluded',
                                    'reason': 'Pertanyaan tidak menyebut keadaan atau peristiwa rupiah yang '
                                              'ingin dijelaskan; alasan penurunan, nama mata uang, atau '
                                              'makna lain hanya dapat ditebak.'},
 'query_c2afb5dc724caf4a703b1d06': {'query_text': 'Apa arti dari rupiah?',
                                    'domain': 'keuangan',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta pengertian rupiah sebagai istilah mata uang '
                                              'Indonesia; intent definisi keuangan jelas.'},
 'query_9a25b5807d822806812e3d55': {'query_text': 'Kapan rupiah paling kuat?',
                                    'domain': 'keuangan',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta penjelasan historis periode kekuatan rupiah; tolok '
                                              'ukur dan pembanding dapat dijelaskan dalam jawaban, tanpa '
                                              'mengasumsikan satu ukuran kekuatan universal.'},
 'query_ca0014c0abe189b7ef440466': {'query_text': '1 rupiah apakah ada?',
                                    'domain': 'keuangan',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta informasi keberadaan nominal satu rupiah; jawaban '
                                              'dapat membedakan keberadaan historis dan penggunaan saat '
                                              'ini.'},
 'query_7889d6037a6816da92364f40': {'query_text': 'Form seperti apa?',
                                    'domain': 'teknologi',
                                    'language': 'id',
                                    'status': 'excluded',
                                    'reason': 'Tidak menyebut jenis form atau konteks digital; dapat merujuk '
                                              'formulir fisik, digital, atau bentuk lain sehingga objek '
                                              'teknologi perlu ditebak.'},
 'query_cef09767e7698a0715153bd4': {'query_text': 'Apa situs web untuk membuat formulir?',
                                    'domain': 'teknologi',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta pilihan situs web pembuat formulir; konteks teknologi '
                                              'dan kebutuhan informasi alat digital eksplisit.'},
 'query_041fcd902a7d4e7c58a019aa': {'query_text': 'Form untuk apa?',
                                    'domain': 'teknologi',
                                    'language': 'id',
                                    'status': 'excluded',
                                    'reason': 'Objek form tidak dibatasi pada formulir digital atau '
                                              'aplikasi; menerima sebagai query teknologi memerlukan konteks '
                                              'keyword asal.'},
 'query_1849f4d0dc669879aec74bf6': {'query_text': 'Form apa saja?',
                                    'domain': 'teknologi',
                                    'language': 'id',
                                    'status': 'excluded',
                                    'reason': 'Kategori form yang hendak didaftar tidak ditentukan dan '
                                              'konteks teknologi tidak terlihat dari teks pertanyaan.'},
 'query_7b0c3a040058537e2d189937': {'query_text': 'Kapan jadwal penukaran uang baru 2026?',
                                    'domain': 'keuangan',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta jadwal penukaran uang baru pada tahun tertentu; '
                                              'berbeda dari prosedur penukaran atau ketersediaan layanan di '
                                              'bank tertentu.'},
 'query_44be37be1976dd1045d94fac': {'query_text': 'Apa yang dimaksud dengan uang?',
                                    'domain': 'keuangan',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta definisi uang secara umum; cakupan lebih luas '
                                              'daripada definisi rupiah sebagai mata uang tertentu.'},
 'query_25d01d39d78727a85b83d72d': {'query_text': 'Kapan uang 1.000 jadi 1 rupiah?',
                                    'domain': 'keuangan',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta informasi waktu perubahan nominal seribu menjadi satu '
                                              'rupiah; premis perubahan tersebut boleh diperiksa atau '
                                              'dikoreksi tanpa dianggap fakta peneliti.'},
 'query_1a07402e9ee8c703b459a1c0': {'query_text': 'Berapa harga iPhone 14 di iBox?',
                                    'domain': 'teknologi',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta harga perangkat iPhone 14 pada penjual tertentu; '
                                              'model perangkat dan sumber harga jelas, variasi kapasitas '
                                              'dapat dijelaskan.'},
 'query_a3d06670fd3f6ed296b4900f': {'query_text': 'iBox Indonesia milik siapa?',
                                    'domain': 'teknologi',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta identitas pemilik iBox Indonesia sebagai sumber '
                                              'penjualan perangkat teknologi; diterima sebagai informasi '
                                              'institusi dalam ekosistem produk teknologi.'},
 'query_505362b08364f5c7fca34e59': {'query_text': 'iPhone 11 iBox harganya berapa?',
                                    'domain': 'teknologi',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta harga iPhone 11 melalui iBox; model perangkat '
                                              'membedakan kebutuhan informasi dari iPhone 14.'},
 'query_0cb4cb3d91c767c999743ab3': {'query_text': 'iPhone iBox itu apa sih?',
                                    'domain': 'teknologi',
                                    'language': 'id',
                                    'status': 'accepted',
                                    'reason': 'Meminta penjelasan istilah iPhone iBox dalam pembelian '
                                              'perangkat; objek produk teknologi dan intent definisi jelas.'}}


In [ ]:
# Terapkan hanya ke query yang belum selesai ditinjau atau masih milik review asisten ini.
# Keputusan peneliti serta keputusan final lain tidak ditimpa.
REVIEWER = 'asisten_rubrik_target300_v1'
current_rows = read_rows(OUTPUT / 'candidates.csv')
current_by_id = {r['query_id']: r for r in current_rows}
manual_rows = read_rows(ROOT / 'data/manual' / (config['expansion_id'] + '_decisions.csv'))
manual_by_id = {r['query_id']: r for r in manual_rows}
assistant_changes = {}
skipped = []
for query_id, decision in paa_target_decisions.items():
    row = current_by_id.get(query_id)
    if row is None or row['query_text'] != decision['query_text'] or row['domain'] != decision['domain']:
        raise ValueError(f'Sumber review berubah atau tidak ditemukan: {query_id}')
    previous = manual_by_id.get(query_id, {})
    unresolved = row['selection_status'] in {'pending', 'needs_review'}
    owned = previous.get('reviewer') == REVIEWER
    if not (unresolved or owned):
        skipped.append(decision['query_text'])
        continue
    change = {k: decision[k] for k in ('language', 'status', 'reason')}
    change['reviewer'] = REVIEWER
    if any(previous.get(k) != v for k, v in change.items()):
        assistant_changes[query_id] = change
if assistant_changes:
    candidates = prepare(config, assistant_changes)
else:
    candidates = current_rows
    print('Tidak ada perubahan keputusan asisten yang perlu disimpan.')
print('Keputusan asisten diperbarui:', len(assistant_changes))
print('Keputusan final lain dipertahankan:', len(skipped))
print('Status kandidat terbaru:', dict(Counter(r['selection_status'] for r in candidates)))
show([current_by_id[qid] | {'review_status': d['status'], 'review_reason': d['reason']}
      for qid, d in paa_target_decisions.items()],
     ['query_text', 'domain', 'review_status', 'review_reason'], limit=len(paa_target_decisions))


## Koreksi keputusan oleh peneliti

Gunakan `decisions_by_text` untuk mengganti keputusan query tertentu dengan teks persis dari tabel. Bahasa `id` wajib untuk penerimaan. Alasan adalah metadata review dan tidak dikirim sebagai tambahan prompt Gemini.

Enam keputusan yang sudah kamu isi tetap dipertahankan. Contoh **Driver apa saja?** sekarang berstatus `excluded` karena ambigu antara perangkat lunak dan pengemudi ketika dibaca tanpa konteks tambahan. Pertanyaan **5 bank BUMN apa saja?** diterima karena intent daftar bank jelas meskipun premis jumlah boleh dikoreksi oleh jawaban.

Salin entri contoh untuk koreksi lainnya, lalu isi status dan alasan. Jalankan sel untuk menyimpan keputusan sebagai `reviewer=peneliti`; hasilnya mengungguli keputusan asisten pada sel sebelumnya. Sel terakhir menampilkan daftar diterima terbaru.


In [23]:
decisions_by_text = {
    'Driver apa saja?': {
        'language': 'id',
        'status': 'excluded',
        'reason': (
            'kata driver ambigu'
        ),
    },
    'Apakah anemia bisa menyebabkan jantung?': {
        'language': 'id',
        'status': 'excluded',
        'reason': (
            'kondisi jantung yang ditanyakan tidak lengkap.'
        ),
    },
    '5 bank BUMN apa saja?': {
        'language': 'id',
        'status': 'accepted',
        'reason': (
            'pertanyaan jelas, hanya meminta menyebtukan 5 buah bank dari BUMN'
        ),
    },
    'Kenapa harga saham anjlok?': {
        'language': 'id',
        'status': 'accepted',
        'reason': (
            'pertanyaan jelas, hanya meminta penjelasan bagaimana bisa harga saham bisa turun.'
        ),
    },
    '1 lot saham harga berapa?': {
        'language': 'id',
        'status': 'accepted',
        'reason': (
            'Intent jelas, menanyakan biaya membeli satu lot saham. Karena emiten dan waktu tidak disebutkan, pertanyaan diterima sebagai permintaan penjelasan umum'
        ),
    },
    '1 lot saham bisa untung berapa?': {
        'language': 'id',
        'status': 'accepted',
        'reason': (
            'jawaban dapat menjelaskan faktor yang memengaruhi hasil'
        ),
    },
    # 'Teks persis pertanyaan': {
    #     'language': 'id', 'status': 'accepted',
    #     'reason': 'Alasan keputusan setelah saya tinjau.'
    # },
}
changes = {}
for text, decision in decisions_by_text.items():
    matches = [r for r in candidates if r['query_text'] == text]
    if len(matches) != 1:
        raise ValueError(f'Teks tidak ditemukan atau ambigu: {text}')
    changes[matches[0]['query_id']] = {**decision, 'reviewer': 'peneliti'}
if changes:
    candidates = prepare(config, changes)
else:
    print('Tidak ada koreksi; keputusan tersimpan dipertahankan.')

Kemunculan PAA: 324 | query unik dengan domain: 316
Kandidat baru: 309 {'kesehatan': 121, 'keuangan': 109, 'teknologi': 79}
Status: {'accepted': 246, 'excluded': 63}
Output: D:\Kuliah\TA\final-assignment\data\interim\query_expansion\query_expansion_01


In [24]:
accepted = read_rows(OUTPUT / 'accepted_new.csv')
print('Query tambahan diterima:', len(accepted))
print('Berkas untuk batch berikutnya:', OUTPUT / 'accepted_new.csv')
show(accepted, ['query_id', 'query_text', 'domain', 'topic_id', 'source_batch', 'parent_query_id'], limit=30)

Query tambahan diterima: 246
Berkas untuk batch berikutnya: d:\Kuliah\TA\final-assignment\data\interim\query_expansion\query_expansion_01\accepted_new.csv


query_id,query_text,domain,topic_id,source_batch,parent_query_id
query_0635994cd811497621863283,Apa sih ciri-ciri penyakit jantung?,kesehatan,topic_b477da794ae11c5a,paa_expansion_01,
query_18166133111fd1748aeaabd3,Rebusan kunyit apakah baik untuk jantung?,kesehatan,topic_b477da794ae11c5a,paa_expansion_01,
query_47a542981a7fa202964c8f29,Apakah penyakit jantung bisa menyebabkan batuk?,kesehatan,topic_b477da794ae11c5a,paa_expansion_01,
query_03748f11ccd253be1388abcc,Paracetamol adalah obat untuk penyakit apa?,kesehatan,topic_54238e9415df5dff,paa_expansion_01,
query_5a5fbb8f174324e3af98e565,Berapa dosis parasetamol yang aman untuk ibu hamil?,kesehatan,topic_54238e9415df5dff,paa_expansion_01,
query_71e79135aef475fd88221559,Paracetamol apakah aman untuk ibu menyusui?,kesehatan,topic_54238e9415df5dff,paa_expansion_01,
query_a71001b8bd74a719f78eb6d0,Kista itu penyakit apa sih?,kesehatan,topic_9872adb577a7de8e,paa_expansion_01,
query_47b6e8611be975334b6c2b12,Apakah kista itu berbahaya?,kesehatan,topic_9872adb577a7de8e,paa_expansion_01,
query_aa6f9dde188807f9e6c0de68,Kista bisa hilang dengan apa?,kesehatan,topic_9872adb577a7de8e,paa_expansion_01,
query_f5ae9c1eac428ffd58aa554f,Apa ciri-ciri kista?,kesehatan,topic_9872adb577a7de8e,paa_expansion_01,


Ditampilkan 30 dari 246 baris.
